# Benchmark: três versões da distância euclidiana

**Fase 1, Semana 1, Sessão 3.** Compara `euclidean_two_loops`, `euclidean_one_loop` e
`euclidean_vectorized` (de `mlscratch.utils.distances`) com `sklearn.metrics.pairwise_distances`.

Protocolo do roteiro:
1. **Paridade:** os resultados batem (`np.allclose`)?
2. **Tempo:** `timeit` com n = m ∈ {100, 500, 2000} e d = 50.
3. **Escalabilidade:** gráfico log-log de tempo × n.
4. **Diferenças de implementação:** o que o sklearn faz que a minha versão não faz.

Meta do mini-projeto: a vetorizada pelo menos **50× mais rápida** que a de dois loops em n = 2000.

> A versão de dois loops leva ~15 s por chamada em n = 2000; a execução completa demora cerca de 1 minuto.

In [ ]:
import timeit

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import pairwise_distances

from mlscratch.utils.distances import (
    euclidean_one_loop,
    euclidean_two_loops,
    euclidean_vectorized,
)

rng = np.random.default_rng(42)

implementacoes = {
    "dois loops": euclidean_two_loops,
    "um loop": euclidean_one_loop,
    "vetorizada": euclidean_vectorized,
    "sklearn": lambda A, B: pairwise_distances(A, B, metric="euclidean"),
}

## 1. Paridade

Antes de medir tempo, confirmar que todas calculam a mesma coisa.

In [ ]:
A = rng.normal(size=(300, 50))
B = rng.normal(size=(200, 50))
referencia = pairwise_distances(A, B)

for nome, f in implementacoes.items():
    D = f(A, B)
    erro_max = np.abs(D - referencia).max()
    print(f"{nome:<11} shape={D.shape}  allclose={np.allclose(D, referencia)}  erro máx={erro_max:.2e}")

Caso de borda: distância de um ponto para ele mesmo. Pela identidade
‖a‖² + ‖b‖² − 2aᵀb, o arredondamento pode deixar `D2` levemente negativo, e é por isso que existe o `np.maximum(D2, 0)`.

In [ ]:
X = rng.normal(size=(500, 50)) * 1e3
D2_bruto = np.sum(X**2, axis=1)[:, None] + np.sum(X**2, axis=1) - 2 * X @ X.T
print("menor valor de D2 sem o clip:", D2_bruto.min())
print("diagonal da vetorizada (deveria ser 0):", np.abs(np.diag(euclidean_vectorized(X, X))).max())

## 2. Tempo com `timeit`

`timeit.repeat` roda a função `repeat` vezes e usamos a **mediana**, que é menos sensível a ruído
(outros processos, cache) que a média. As versões com loop rodam menos vezes porque são lentas.

In [ ]:
ns = [100, 500, 2000]
d = 50
repeticoes = {"dois loops": 3, "um loop": 5, "vetorizada": 20, "sklearn": 20}

tempos = {nome: [] for nome in implementacoes}
for n in ns:
    A = rng.normal(size=(n, d))
    B = rng.normal(size=(n, d))
    for nome, f in implementacoes.items():
        t = timeit.repeat(lambda: f(A, B), number=1, repeat=repeticoes[nome])
        tempos[nome].append(np.median(t))
    print(f"n = {n} ok")

In [ ]:
print(f"{'implementação':<14}" + "".join(f"{'n=' + str(n):>12}" for n in ns))
for nome, ts in tempos.items():
    print(f"{nome:<14}" + "".join(f"{t * 1e3:>10.2f}ms" for t in ts))

print()
base = tempos["dois loops"][-1]
for nome, ts in tempos.items():
    print(f"speedup vs. dois loops em n=2000 — {nome:<11}: {base / ts[-1]:>8.1f}×")

In [ ]:
speedup = tempos["dois loops"][-1] / tempos["vetorizada"][-1]
assert speedup >= 50, f"meta não atingida: {speedup:.1f}×"
print(f"Meta atingida: vetorizada {speedup:.0f}× mais rápida que dois loops em n=2000")

## 3. Escalabilidade (log-log)

Em escala log-log, um custo O(n²) vira uma reta de inclinação 2. As quatro versões fazem O(n² · d)
operações; o que muda é a **constante** (o overhead do Python por iteração), que aparece como
deslocamento vertical entre as retas.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for nome, ts in tempos.items():
    ax.plot(ns, ts, marker="o", label=nome)

# Reta de referência O(n²), ancorada na vetorizada
ref = tempos["vetorizada"][0] * (np.array(ns) / ns[0]) ** 2
ax.plot(ns, ref, "k--", alpha=0.4, label="O(n²) referência")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("n = m (número de pontos)")
ax.set_ylabel("tempo (s, mediana)")
ax.set_title(f"Distância euclidiana par-a-par (d = {d})")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()

fig.savefig("../../benchmarks/figures/distancias_loglog.png", dpi=150)
plt.show()

In [ ]:
# Inclinação estimada de cada curva (≈ 2 esperado para O(n²))
for nome, ts in tempos.items():
    inclinacao = np.polyfit(np.log(ns), np.log(ts), 1)[0]
    print(f"{nome:<11}: inclinação ≈ {inclinacao:.2f}")

## 4. Conclusões (preencher depois de rodar)

**Paridade:** 

**Tempos (n = 2000):** 

**Por que a vetorizada é tão mais rápida?** 

**Por que as inclinações dão ≈ 2 (ou não)?** 

**O que o sklearn faz diferente da minha versão?** (dica: leia a docstring de
`sklearn.metrics.pairwise.euclidean_distances`; procure por precisão em float32, matrizes esparsas e `n_jobs`)

**O que aprendi e não esperava:** 